# **LangChain Chaining**

In [1]:
!pip install langchain==0.3.28 langchain-google-genai langchain-community langchain-core numpy==1.26.4 wikipedia  ddgs

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
!pip install grandalf

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 31.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
import os
import getpass
from langchain.llms import Ollama
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
 if not os.environ.get("GOOGLE_API_KEY"):
   os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

Enter API key for Google Gemini:  ·······································


In [3]:
# LLM
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
ollm=Ollama(
    model="llama3.1:8b",
    temperature=0,
)

# Simple Chain

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

# model = ChatOpenAI()
model = llm

parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)
chain.get_graph().print_ascii()

A great request! Here are 5 interesting facts about cricket:

1. **The Longest Match**: The longest-ever Test match in cricket was played between England and Australia from August 31 to September 7, 1939, at the Old Trafford Cricket Ground in Manchester, England. The match lasted for eight days (yes, you read that right - eight days!) with a total of 974.1 overs bowled.
2. **The Fastest Bowler**: The fastest recorded delivery in cricket was bowled by Shoaib Akhtar of Pakistan on January 20, 2003. His delivery clocked an astonishing speed of 161.3 km/h (100 mph) at the Wanderers Stadium in Johannesburg, South Africa.
3. **The First ODI**: The first One-Day International (ODI) cricket match was played between England and Australia on January 5, 1971, at the Melbourne Cricket Ground (MCG). This game introduced the concept of limited-overs cricket, where teams face a fixed number of overs in a single day.
4. **The Wicketkeeper's Record**: The most dismissals by a wicketkeeper in Test crick

# Sequential Chain

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model =llm

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic': 'Unemployment in India'})

print(result)
chain.get_graph().print_ascii()

Here is a 5-pointer summary of the report on Unemployment in India:

**1. Unemployment Rates:** The unemployment rate in India is estimated to be around 6-7%, with a higher rate among young people aged 15-24 (18-20%).

**2. Types and Causes of Unemployment:** There are four types of unemployment: Voluntary, Involuntary, Structural, and Frictional. Key causes include Lack of Job Creation, Limited Vocational Training, Rapid Urbanization, and Social and Cultural Barriers.

**3. Consequences of Unemployment:** High levels of unemployment contribute to poverty, social exclusion, mental health issues, and economic inefficiencies.

**4. Policy Recommendations:** To address unemployment, the report recommends investing in education and skills training, promoting entrepreneurship, increasing job creation, strengthening labor market institutions, and implementing social protection programs.

**5. Future Research Directions:** The report suggests conducting further research on estimating unemploy

# Parallel Chain

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel


model1 = llm

model2 = llm

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print(result)
chain.get_graph().print_ascii()

Here is the merged document:

**Support Vector Machines (SVMs)**

**Advantages of Support Vector Machines (SVMs)**

* Effective in high-dimensional spaces
* Memory efficient due to using support vectors only
* Versatile with various kernel functions available

**Disadvantages of SVMs**

* Choosing correct kernel function and regularization term is crucial to avoid over-fitting
* Does not provide direct probability estimates (calculated through cross-validation)

**Optimal Input Format for SVMs**

* Dense: C-ordered numpy.ndarray or scipy.sparse.csr_matrix with dtype=float64

**Frequently Asked Questions about Support Vector Machines (SVMs)**

**Q1: What is the primary purpose of Support Vector Machines (SVMs)?**
A1: Classification, regression, and outliers detection.

**Q2: Why are SVMs effective in high-dimensional spaces?**
A2: Because they use a subset of training points in the decision function.

**Q3: What can happen if the number of features is much greater than the number of sam

# Conditonal Chain

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal


parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

chain.get_graph().print_ascii()

Here's a possible response:

"Thank you so much for your kind words! I'm glad you appreciated [specific aspect of the interaction or product]. Your support means a lot to me!"

Or, if it was on social media:

"Wow, thank you for sharing such positive feedback! I'm thrilled that you enjoyed [experience/product] and can't wait to share more with you in the future!"
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
       +--------+        
       | Ollama |        
       +--------+        
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--